# Position QA — Spot Check

Compare **trained** vs **untrained** Qwen3-VL-2B-Instruct on spatial reasoning:
given a game screenshot, answer questions like "Is the gold to the left or right of you?"

- **Trained**: checkpoint-28000 (merged full model)
- **Untrained**: base Qwen3-VL-2B-Instruct from HuggingFace

In [ ]:
import sys
sys.path.insert(0, "..")

import random
import torch
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq

CHECKPOINT_DIR = "../outputs/position_qa_dpo/merged_checkpoint_28000"
BASE_MODEL = "Qwen/Qwen3-VL-2B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(42)

In [ ]:
# Generate test samples using the same PositionQAGenerator as training
from src.data.position_qa_generator import PositionQAGenerator

generator = PositionQAGenerator(cross_axis_negative_prob=0.0)  # no cross-axis for cleaner eval
test_samples = generator.generate_batch(12)  # 12 examples
print(f"Generated {len(test_samples)} test samples")

In [ ]:
# Load processor (from checkpoint so chat format matches)
processor = AutoProcessor.from_pretrained(CHECKPOINT_DIR, trust_remote_code=True)

In [ ]:
# Load base model (untrained)
base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()
print("Base model loaded")

In [ ]:
# Load trained model (merged checkpoint)
trained_model = AutoModelForVision2Seq.from_pretrained(
    CHECKPOINT_DIR,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
trained_model.eval()
print("Trained model loaded from", CHECKPOINT_DIR)

In [ ]:
def generate_response(model, processor, image, prompt, max_new_tokens=32):
    """Generate response for image + prompt. Uses same format as trainer."""
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    ]
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    generated = out[0][prompt_len:]
    return processor.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)

In [ ]:
def is_correct(response, chosen):
    """Check if response is semantically correct (contains the key direction)."""
    r = response.strip().lower()
    c = chosen.strip().lower()
    # Key direction words
    if "left" in c or "right" in c:
        return ("left" in r and "left" in c) or ("right" in r and "right" in c)
    if "up" in c or "above" in c:
        return "up" in r or "above" in r
    if "down" in c or "below" in c:
        return "down" in r or "below" in r
    return False

## Run inference and compare

In [ ]:
results = []
for i, sample in enumerate(test_samples):
    img = sample["image"]
    prompt = sample["prompt"]
    chosen = sample["chosen"]
    
    base_out = generate_response(base_model, processor, img, prompt)
    trained_out = generate_response(trained_model, processor, img, prompt)
    
    base_ok = is_correct(base_out, chosen)
    trained_ok = is_correct(trained_out, chosen)
    
    results.append({
        "prompt": prompt,
        "chosen": chosen,
        "base_out": base_out,
        "trained_out": trained_out,
        "base_correct": base_ok,
        "trained_correct": trained_ok,
    })

base_acc = sum(r["base_correct"] for r in results) / len(results)
trained_acc = sum(r["trained_correct"] for r in results) / len(results)
print(f"Base model accuracy:   {base_acc:.1%} ({sum(r['base_correct'] for r in results)}/{len(results)})")
print(f"Trained model accuracy: {trained_acc:.1%} ({sum(r['trained_correct'] for r in results)}/{len(results)})")

## Side-by-side examples

In [ ]:
from IPython.display import display

for i, r in enumerate(results):
    print("=" * 70)
    print(f"Example {i+1}")
    print(f"Prompt:  {r['prompt']}")
    print(f"Correct: {r['chosen']}")
    display(test_samples[i]["image"])
    print(f"  Base:    {r['base_out']} {'✓' if r['base_correct'] else '✗'}")
    print(f"  Trained: {r['trained_out']} {'✓' if r['trained_correct'] else '✗'}")
    print()

In [ ]:
# Text-only comparison (use if image display fails)
for i, r in enumerate(results):
    print("=" * 70)
    print(f"Example {i+1} | Prompt: {r['prompt']} | Correct: {r['chosen']}")
    print(f"  Base: {r['base_out']} {'✓' if r['base_correct'] else '✗'}")
    print(f"  Trained: {r['trained_out']} {'✓' if r['trained_correct'] else '✗'}")